In [1]:
import csv
import json
import os
import shutil
from collections import Counter, defaultdict

ID_SUM = 29
SEED = 9
RECORDS_TO_PROCESS = 1000 + ID_SUM
TARGET_CLASS_PREFIX = str((SEED % 4) + 2)



In [ ]:
input_file = "weblog.csv"
output_dir = "output"
os.makedirs(output_dir, exist_ok=True)

filtered_csv_path = os.path.join(output_dir, "filtered_weblog.csv")
json_summary_path = os.path.join(output_dir, "log_summary.json")
backup_json_path = os.path.join(output_dir, "log_summary_backup.json")

processed_count = 0
target_class_count = 0
status_code_counts = Counter()
url_counts_in_target = Counter()
filtered_rows = []

print(f"Starting processing: Target status class is {TARGET_CLASS_PREFIX}xx, processing {RECORDS_TO_PROCESS} records...")

Starting processing: Target status class is 3xx, processing 1029 records...


In [17]:
with open(input_file, mode="r", encoding="utf-8") as f:
    reader = csv.reader(f)
    header = next(reader, None)

    for row in reader:
        if processed_count >= RECORDS_TO_PROCESS:
            break

        try:
            ip = row[0].strip()
            timestamp = row[1].strip()
            url = row[2].strip()
            status = row[3].strip()

            if not ip or not status.isdigit():
                continue

            processed_count += 1
            status_code_counts[status] += 1

            if status.startswith(TARGET_CLASS_PREFIX):
                target_class_count += 1
                url_counts_in_target[url] += 1
                filtered_rows.append((ip, timestamp, url, status))

        except (IndexError, ValueError):
            continue

    top_5_urls = url_counts_in_target.most_common(5)

    with open(filtered_csv_path, mode="w", newline="", encoding="utf-8") as f_out:
        writer = csv.writer(f_out)
        writer.writerow(["IP", "Time", "URL", "Status"])
        writer.writerows(filtered_rows)

    summary_data = {
        "records_processed": processed_count,
        "target_status_class": f"{TARGET_CLASS_PREFIX}xx",
        "records_in_target_class": target_class_count,
        "count_per_status_code": dict(status_code_counts),
        "five_most_requested_urls": top_5_urls
    }

    with open(json_summary_path, mode="w", encoding="utf-8") as j_out:
        json.dump(summary_data, j_out, indent=4)

    shutil.copy(json_summary_path, backup_json_path)
    backup_exists = os.path.exists(backup_json_path)

In [20]:
print("\n--- Processing Complete ---")
print(f"Records Processed: {processed_count}")
print(f"Target Status Class: {TARGET_CLASS_PREFIX}xx")
print(f"Records in Target Class: {target_class_count}")
print(f"Count per status code: {status_code_counts}")
print(f"Filtered CSV saved to: {filtered_csv_path}")
print(f"JSON Summary saved to: {json_summary_path}")
print(f"Backup Verification Exists: {backup_exists}")


--- Processing Complete ---
Records Processed: 1029
Target Status Class: 3xx
Records in Target Class: 166
Count per status code: Counter({'200': 850, '302': 145, '304': 21, '206': 8, '404': 5})
Filtered CSV saved to: output\filtered_weblog.csv
JSON Summary saved to: output\log_summary.json
Backup Verification Exists: True
